# 02 - Data cleaning

Build a typed dataset for analysis using the same pipeline as the command-line export. Parquet preserves types and missing values for later stages.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project = next(
    folder for folder in (cwd, *cwd.parents)
    if (folder / "src" / "preprocessing.py").is_file()
)
sys.path.insert(0, str(project))
from src.preprocessing import clean_tickets, read_raw_tickets, write_processed_tickets

raw = read_raw_tickets(project.parent / "tickets.csv")

## Evaluate the free-text fields

Measure completeness, repetition and length without displaying the contents. The first model will use structured fields. Descriptions can be evaluated later for a text-based model after reviewing their content and handling personal information. Resolution notes are produced after opening.

In [ ]:
text_columns = ["issue_description", "resolution_notes"]
text = raw[text_columns].apply(lambda column: column.str.strip())
text = text.mask(text.eq(""))
display(pd.DataFrame({
    "missing": text.isna().sum(),
    "distinct_nonmissing": text.nunique(),
    "median_length": text.apply(lambda column: column.str.len().median()),
}))

## Apply cleaning rules

Remove exact duplicate source records and check that each remaining ticket ID is present and unique. Trim whitespace and convert blank cells to missing values. Convert date, numeric, category, and yes/no fields to explicit types. Keep missing resolution times and satisfaction scores as missing because a fill value would misrepresent the outcome.

Keep recorded durations and flag inconsistencies. For example, a final resolution before creation conflicts with the event order, while an unusually long duration may be a genuine long-running case. A flag indicates that the check fired. A false value can also mean there was too little information to trigger the check.

In [ ]:
tickets = clean_tickets(raw)
print(f"Source rows: {len(raw):,} | Cleaned rows: {len(tickets):,}")
display(tickets.dtypes.astype(str).value_counts().rename("columns").to_frame())
missing = tickets.isna().sum()
display(missing[missing.gt(0)].sort_values(ascending=False).rename("missing").to_frame())
display(tickets.filter(regex="^flag_").sum().rename("tickets").to_frame())

## Save and verify

Write the local processed dataset. Read it back and compare it to the in-memory result, including dtypes and missing values.

In [ ]:
output = project / "data" / "processed" / "tickets.parquet"
write_processed_tickets(tickets, output)
restored = pd.read_parquet(output)
pd.testing.assert_frame_equal(tickets, restored)
print(f"Verified {len(restored):,} rows and {len(restored.columns)} columns at {output}")